# Merge multihop benchmark sources into the Wikidata schema

This notebook merges benchmark examples from several sources into one dataset with the same top-level schema as the Wikidata JSONL file.

It is intentionally conservative: source files are read-only, the final output is written to a new file, and existing output files are not overwritten unless `OVERWRITE_OUTPUT = True`.

## Contents

1. [Settings](#settings)
2. [Imports](#imports)
3. [I/O helpers](#io-helpers)
4. [Schema helpers](#schema-helpers)
5. [Converters](#converters)
6. [Load source records](#load-source-records)
7. [Convert and merge](#convert-and-merge)
8. [Validate and inspect](#validate-and-inspect)
9. [Save final dataset](#save-final-dataset)


<a id="settings"></a>

## 1. Settings

In [1]:
from pathlib import Path

# Run this notebook from the repository root: multihop_benchmark/
BASE_DIR = Path(".").resolve()

SOURCE_PATHS = {
    "wikidata": BASE_DIR / "wikidata" / "out_wikidata_benchmark" / "dataset_main.full_gold_enriched.jsonl",
    "zero_gold": BASE_DIR / "wikidata" / "out_wikidata_benchmark" / "zero_gold.json",
    "kqapro": BASE_DIR / "kqapro" / "artifacts_kqapro_stage6" / "jsonl" / "kqapro_selected_ru_final.jsonl",
    "mintaka": BASE_DIR / "mintaka" / "artifacts_mintaka_stage3" / "jsonl" / "mintaka_selected_ru_final.jsonl",
}

OUTPUT_DIR = BASE_DIR / "merged_dataset"
OUTPUT_JSON = OUTPUT_DIR / "multihop_benchmark_merged_wikidata_format.json"
OUTPUT_JSONL = OUTPUT_DIR / "multihop_benchmark_merged_wikidata_format.jsonl"
DIAGNOSTICS_CSV = OUTPUT_DIR / "multihop_benchmark_merged_diagnostics.csv"

# Safety switch. Keep False by default: the notebook will create a versioned file if the target exists.
OVERWRITE_OUTPUT = False

# Dedupe policy. Query-level deduplication is usually the safest for final benchmark assembly.
DEDUP_BY_QUERY_TEXT = True
DEDUP_BY_ID = True

# Store the full original non-Wikidata record inside constraints["_source_record"].
# This keeps metadata without adding unpredictable top-level fields.
KEEP_ORIGINAL_RECORD_IN_CONSTRAINTS = True


<a id="imports"></a>

## 2. Imports

In [2]:
import json
import hashlib
import datetime as dt
from copy import deepcopy
from collections import Counter, defaultdict
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd


<a id="io-helpers"></a>

## 3. I/O helpers

In [3]:
def utc_now_z() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat().replace("+00:00", "Z")


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"Broken JSONL line {line_no} in {path}: {e}") from e
            if not isinstance(obj, dict):
                raise ValueError(f"Expected object at line {line_no} in {path}, got {type(obj).__name__}")
            records.append(obj)
    return records


def _records_from_json_obj(obj: Any, path: Path) -> List[Dict[str, Any]]:
    if isinstance(obj, list):
        if not all(isinstance(x, dict) for x in obj):
            raise ValueError(f"Expected a list of objects in {path}")
        return obj

    if isinstance(obj, dict):
        # Common wrappers used by preprocessing notebooks.
        for key in ("records", "examples", "data", "items", "questions", "rows"):
            value = obj.get(key)
            if isinstance(value, list) and all(isinstance(x, dict) for x in value):
                return value
        # A single JSON object is treated as one benchmark record.
        return [obj]

    raise ValueError(f"Unsupported JSON root in {path}: {type(obj).__name__}")


def read_json_or_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(path)

    if path.suffix.lower() == ".jsonl":
        return read_jsonl(path)

    with path.open("r", encoding="utf-8") as f:
        obj = json.load(f)
    return _records_from_json_obj(obj, path)


def safe_output_path(path: Path, overwrite: bool = False) -> Path:
    if overwrite or not path.exists():
        return path

    stem, suffix = path.stem, path.suffix
    for i in range(1, 10_000):
        candidate = path.with_name(f"{stem}.v{i:03d}{suffix}")
        if not candidate.exists():
            return candidate
    raise RuntimeError(f"Could not find a free versioned path for {path}")


def write_jsonl(records: List[Dict[str, Any]], path: Path, overwrite: bool = False) -> Path:
    path = safe_output_path(path, overwrite=overwrite)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    return path


def write_json_array(records: List[Dict[str, Any]], path: Path, overwrite: bool = False) -> Path:
    path = safe_output_path(path, overwrite=overwrite)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    return path


<a id="schema-helpers"></a>

## 4. Schema helpers

In [4]:
DEFAULT_TARGET_FIELDS = [
    "id",
    "domain",
    "complexity",
    "query_text_ru",
    "constraints",
    "requested_count",
    "gold_answer_qids",
    "gold_answer_labels_ru",
    "sparql_query",
    "created_at",
    "is_advanced",
    "template_id",
    "template_family",
    "gold_truncated",
    "ask_validator_sparql",
]

FIELD_DEFAULTS = {
    "id": "",
    "domain": "unknown",
    "complexity": "unknown",
    "query_text_ru": "",
    "constraints": {},
    "requested_count": 0,
    "gold_answer_qids": [],
    "gold_answer_labels_ru": [],
    "sparql_query": "",
    "created_at": "",
    "is_advanced": False,
    "template_id": None,
    "template_family": "default",
    "gold_truncated": False,
    "ask_validator_sparql": None,
}


def stable_hash(text: str, n: int = 12) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()[:n]


def first_present(record: Dict[str, Any], keys: Iterable[str], default: Any = None) -> Any:
    for key in keys:
        if key in record and record[key] not in (None, ""):
            return record[key]
    return default


def as_list(value: Any) -> List[Any]:
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, set):
        return list(value)
    return [value]


def clean_str_list(value: Any) -> List[str]:
    out = []
    for x in as_list(value):
        if isinstance(x, dict):
            x = first_present(x, ["label_ru", "ru", "label", "name", "title", "value", "answer"], default=None)
        if x is None:
            continue
        s = str(x).strip()
        if s and s not in out:
            out.append(s)
    return out


def clean_qid_list(value: Any) -> List[str]:
    out = []
    for x in as_list(value):
        if isinstance(x, dict):
            x = first_present(x, ["qid", "id", "entity_id", "wikidata_id", "value"], default=None)
        if x is None:
            continue
        s = str(x).strip()
        if s and s not in out:
            out.append(s)
    return out


def infer_target_fields(wikidata_records: List[Dict[str, Any]]) -> List[str]:
    if wikidata_records:
        # Preserve exactly the top-level layout of the real Wikidata enriched file.
        fields = list(wikidata_records[0].keys())
        for f in DEFAULT_TARGET_FIELDS:
            if f not in fields:
                fields.append(f)
        return fields
    return DEFAULT_TARGET_FIELDS.copy()


def default_for_field(field: str) -> Any:
    return deepcopy(FIELD_DEFAULTS.get(field, None))


def align_to_target_schema(record: Dict[str, Any], target_fields: List[str]) -> Dict[str, Any]:
    aligned = {}
    for field in target_fields:
        if field in record:
            aligned[field] = record[field]
        else:
            aligned[field] = default_for_field(field)
    return aligned


def normalize_common_types(record: Dict[str, Any]) -> Dict[str, Any]:
    rec = deepcopy(record)
    rec["id"] = str(rec.get("id") or stable_hash(rec.get("query_text_ru", "")))
    rec["domain"] = str(rec.get("domain") or "unknown")
    rec["complexity"] = str(rec.get("complexity") or "unknown")
    rec["query_text_ru"] = str(rec.get("query_text_ru") or "").strip()
    rec["constraints"] = rec.get("constraints") if isinstance(rec.get("constraints"), dict) else {}
    rec["requested_count"] = int(rec.get("requested_count") or 0)
    rec["gold_answer_qids"] = clean_qid_list(rec.get("gold_answer_qids"))
    rec["gold_answer_labels_ru"] = clean_str_list(rec.get("gold_answer_labels_ru"))
    rec["sparql_query"] = str(rec.get("sparql_query") or "")
    rec["created_at"] = str(rec.get("created_at") or utc_now_z())
    rec["is_advanced"] = bool(rec.get("is_advanced", False))
    rec["gold_truncated"] = bool(rec.get("gold_truncated", False))
    return rec


<a id="converters"></a>

## 5. Converters

In [5]:
QUERY_KEYS = [
    "query_text_ru", "question_ru_manual", "question_ru", "query_ru", "ru_question",
    "question", "query", "text", "prompt",
]

COMPLEXITY_KEYS = ["complexity", "level", "difficulty", "complexity_level", "hardness", "hop_level"]
DOMAIN_KEYS = ["domain", "source_domain", "category", "topic", "type"]
SPARQL_KEYS = ["sparql_query", "sparql", "query_sparql", "validator_sparql"]
TEMPLATE_KEYS = ["template_id", "template", "template_name"]

GOLD_LABEL_KEYS = [
    "gold_answer_labels_ru", "gold_answer_labels", "gold_labels_ru", "gold_labels",
    "answers_ru", "answer_ru", "answers", "answer", "gold_answers", "gold", "golds",
    "gold_answer", "gold_answer_ru", "gold_answer_label_ru", "gold_answer_label",
]

GOLD_QID_KEYS = [
    "gold_answer_qids", "gold_qids", "answer_qids", "answers_qids", "qids",
    "wikidata_qids", "entity_qids",
]


def extract_query(record: Dict[str, Any]) -> str:
    value = first_present(record, QUERY_KEYS, default="")
    if isinstance(value, dict):
        value = first_present(value, ["ru", "text", "question", "query"], default="")
    return str(value or "").strip()


def extract_complexity(record: Dict[str, Any], source: str) -> str:
    value = first_present(record, COMPLEXITY_KEYS, default=None)
    if value is None:
        if source in {"kqapro", "mintaka"}:
            return "external_multihop"
        if source == "zero_gold":
            return "zero_gold"
        return "unknown"
    value = str(value).strip()
    # Normalize simple integer levels to L1...L5 when possible.
    if value.isdigit() and 1 <= int(value) <= 5:
        return f"L{value}"
    return value


def extract_gold_labels(record: Dict[str, Any]) -> List[str]:
    value = first_present(record, GOLD_LABEL_KEYS, default=[])

    # Some datasets keep answer objects under a nested field.
    if isinstance(value, dict):
        nested = first_present(value, ["ru", "labels_ru", "labels", "answers", "answer", "value"], default=value)
        value = nested

    labels = clean_str_list(value)

    # Additional common nested layouts.
    if not labels:
        for key in ("gold_answers", "answers", "golds"):
            arr = record.get(key)
            if isinstance(arr, list):
                labels = clean_str_list(arr)
                if labels:
                    break
    return labels


def extract_gold_qids(record: Dict[str, Any]) -> List[str]:
    qids = clean_qid_list(first_present(record, GOLD_QID_KEYS, default=[]))
    if qids:
        return qids

    for key in ("gold_answers", "answers", "golds"):
        arr = record.get(key)
        if isinstance(arr, list):
            qids = clean_qid_list(arr)
            if qids:
                return qids
    return []


def base_constraints_from_source(record: Dict[str, Any], source: str) -> Dict[str, Any]:
    constraints = record.get("constraints") if isinstance(record.get("constraints"), dict) else {}
    constraints = deepcopy(constraints)
    constraints.setdefault("source_dataset", source)

    for key in ("original_id", "source_id", "uid", "question_id", "id"):
        if key in record and key != "id":
            constraints.setdefault(key, record[key])

    if KEEP_ORIGINAL_RECORD_IN_CONSTRAINTS and source != "wikidata":
        constraints.setdefault("_source_record", deepcopy(record))

    return constraints


def convert_wikidata_record(record: Dict[str, Any], target_fields: List[str]) -> Dict[str, Any]:
    rec = normalize_common_types(record)
    constraints = rec.get("constraints") if isinstance(rec.get("constraints"), dict) else {}
    constraints.setdefault("source_dataset", "wikidata")
    rec["constraints"] = constraints
    return align_to_target_schema(rec, target_fields)


def convert_zero_gold_record(record: Dict[str, Any], target_fields: List[str]) -> Dict[str, Any]:
    query = extract_query(record)
    source_id = str(first_present(record, ["id", "uid", "question_id"], default=stable_hash(query)))
    rec = {
        "id": f"zero_gold_{source_id}" if not source_id.startswith("zero_gold") else source_id,
        "domain": str(first_present(record, DOMAIN_KEYS, default="zero_gold")),
        "complexity": extract_complexity(record, "zero_gold"),
        "query_text_ru": query,
        "constraints": base_constraints_from_source(record, "zero_gold"),
        "requested_count": 0,
        "gold_answer_qids": [],
        "gold_answer_labels_ru": [],
        "sparql_query": str(first_present(record, SPARQL_KEYS, default="")),
        "created_at": str(first_present(record, ["created_at", "timestamp", "date"], default=utc_now_z())),
        "is_advanced": True,
        "template_id": first_present(record, TEMPLATE_KEYS, default=None),
        "template_family": "zero_gold",
        "gold_truncated": False,
        "ask_validator_sparql": first_present(record, ["ask_validator_sparql", "validator_sparql", "ask_sparql"], default=None),
    }
    return align_to_target_schema(normalize_common_types(rec), target_fields)


def convert_external_record(record: Dict[str, Any], source: str, target_fields: List[str]) -> Dict[str, Any]:
    query = extract_query(record)
    source_id = str(first_present(record, ["id", "uid", "question_id", "original_id"], default=stable_hash(source + "::" + query)))
    labels = extract_gold_labels(record)
    qids = extract_gold_qids(record)

    requested_count = first_present(record, ["requested_count", "num_requested", "answer_count", "n_answers"], default=None)
    if requested_count is None:
        requested_count = len(labels)

    rec = {
        "id": f"{source}_{source_id}" if not source_id.startswith(f"{source}_") else source_id,
        "domain": str(first_present(record, DOMAIN_KEYS, default=source)),
        "complexity": extract_complexity(record, source),
        "query_text_ru": query,
        "constraints": base_constraints_from_source(record, source),
        "requested_count": int(requested_count or 0),
        "gold_answer_qids": qids,
        "gold_answer_labels_ru": labels,
        "sparql_query": str(first_present(record, SPARQL_KEYS, default="")),
        "created_at": str(first_present(record, ["created_at", "timestamp", "date"], default=utc_now_z())),
        "is_advanced": bool(first_present(record, ["is_advanced", "advanced"], default=True)),
        "template_id": first_present(record, TEMPLATE_KEYS, default=None),
        "template_family": source,
        "gold_truncated": bool(first_present(record, ["gold_truncated", "truncated"], default=False)),
        "ask_validator_sparql": first_present(record, ["ask_validator_sparql", "validator_sparql", "ask_sparql"], default=None),
    }
    return align_to_target_schema(normalize_common_types(rec), target_fields)


<a id="load-source-records"></a>

## 6. Load source records

In [6]:
source_records = {}

for source, path in SOURCE_PATHS.items():
    try:
        records = read_json_or_jsonl(path)
        source_records[source] = records
        print(f"{source:10s}: {len(records):5d} records loaded from {path.relative_to(BASE_DIR)}")
    except FileNotFoundError:
        source_records[source] = []
        print(f"{source:10s}: file not found: {path.relative_to(BASE_DIR)}")
    except Exception as e:
        source_records[source] = []
        print(f"{source:10s}: failed to load {path.relative_to(BASE_DIR)}: {e}")

wikidata_records = source_records.get("wikidata", [])
target_fields = infer_target_fields(wikidata_records)

print("\nTarget schema fields:")
print(target_fields)


wikidata  :  1087 records loaded from wikidata/out_wikidata_benchmark/dataset_main.full_gold_enriched.jsonl
zero_gold :    80 records loaded from wikidata/out_wikidata_benchmark/zero_gold.json
kqapro    :    72 records loaded from kqapro/artifacts_kqapro_stage6/jsonl/kqapro_selected_ru_final.jsonl
mintaka   :     9 records loaded from mintaka/artifacts_mintaka_stage3/jsonl/mintaka_selected_ru_final.jsonl

Target schema fields:
['id', 'domain', 'complexity', 'query_text_ru', 'constraints', 'requested_count', 'gold_answer_qids', 'gold_answer_labels_ru', 'sparql_query', 'created_at', 'is_advanced', 'template_id', 'template_family', 'gold_truncated', 'ask_validator_sparql', 'gold_answer_qids_original', 'gold_answer_labels_ru_original', 'gold_old_count', 'gold_enrichment_status', 'gold_enrichment_method', 'gold_enrichment_answer_var', 'gold_enrichment_meta', 'gold_answer_qids_full_raw', 'gold_answer_qids_full', 'gold_answer_labels_ru_full', 'gold_full_count_raw', 'gold_full_count', 'gold_ad

<a id="convert-and-merge"></a>

## 7. Convert and merge

In [7]:
converted_by_source = {}

converted_by_source["wikidata"] = [
    convert_wikidata_record(rec, target_fields)
    for rec in source_records.get("wikidata", [])
]

converted_by_source["zero_gold"] = [
    convert_zero_gold_record(rec, target_fields)
    for rec in source_records.get("zero_gold", [])
]

for source in ("kqapro", "mintaka"):
    converted_by_source[source] = [
        convert_external_record(rec, source, target_fields)
        for rec in source_records.get(source, [])
    ]

all_records = []
for source in ("wikidata", "zero_gold", "kqapro", "mintaka"):
    all_records.extend(converted_by_source[source])
    print(f"{source:10s}: {len(converted_by_source[source]):5d} records after conversion")

print(f"\nTotal before deduplication: {len(all_records)}")


wikidata  :  1087 records after conversion
zero_gold :    80 records after conversion
kqapro    :    72 records after conversion
mintaka   :     9 records after conversion

Total before deduplication: 1248


In [8]:
def deduplicate_records(records: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], Dict[str, int]]:
    kept = []
    seen_ids = set()
    seen_queries = set()
    stats = {"duplicate_id": 0, "duplicate_query_text_ru": 0, "empty_query_text_ru": 0}

    for rec in records:
        rec_id = str(rec.get("id") or "")
        query = str(rec.get("query_text_ru") or "").strip()
        query_key = query.lower()

        if not query:
            stats["empty_query_text_ru"] += 1
            continue

        if DEDUP_BY_ID and rec_id in seen_ids:
            stats["duplicate_id"] += 1
            continue

        if DEDUP_BY_QUERY_TEXT and query_key in seen_queries:
            stats["duplicate_query_text_ru"] += 1
            continue

        kept.append(rec)
        seen_ids.add(rec_id)
        seen_queries.add(query_key)

    return kept, stats

merged_records, dedup_stats = deduplicate_records(all_records)
print(f"Total after deduplication: {len(merged_records)}")
print("Deduplication stats:", dedup_stats)


Total after deduplication: 1196
Deduplication stats: {'duplicate_id': 39, 'duplicate_query_text_ru': 13, 'empty_query_text_ru': 0}


<a id="validate-and-inspect"></a>

## 8. Validate and inspect

In [9]:
def validate_records(records: List[Dict[str, Any]], target_fields: List[str]) -> pd.DataFrame:
    rows = []
    required = ["id", "domain", "complexity", "query_text_ru", "constraints", "requested_count", "gold_answer_qids", "gold_answer_labels_ru", "sparql_query", "created_at"]

    for i, rec in enumerate(records):
        missing = [f for f in required if f not in rec]
        empty_required = [f for f in ("id", "domain", "query_text_ru") if not rec.get(f)]
        extra = [f for f in rec.keys() if f not in target_fields]
        rows.append({
            "row": i,
            "id": rec.get("id"),
            "domain": rec.get("domain"),
            "complexity": rec.get("complexity"),
            "source_dataset": rec.get("constraints", {}).get("source_dataset") if isinstance(rec.get("constraints"), dict) else None,
            "n_gold_qids": len(rec.get("gold_answer_qids") or []),
            "n_gold_labels": len(rec.get("gold_answer_labels_ru") or []),
            "missing_required": ", ".join(missing),
            "empty_required": ", ".join(empty_required),
            "extra_top_level_fields": ", ".join(extra),
        })
    return pd.DataFrame(rows)

validation_df = validate_records(merged_records, target_fields)

print("Rows with missing/empty required fields:")
display(validation_df[(validation_df["missing_required"] != "") | (validation_df["empty_required"] != "")].head(30))

print("\nSource distribution:")
display(validation_df["source_dataset"].value_counts(dropna=False).rename_axis("source").reset_index(name="n"))

print("\nDomain distribution:")
display(validation_df["domain"].value_counts(dropna=False).rename_axis("domain").reset_index(name="n").head(50))

print("\nComplexity distribution:")
display(validation_df["complexity"].value_counts(dropna=False).rename_axis("complexity").reset_index(name="n"))


Rows with missing/empty required fields:


,row,id,domain,complexity,source_dataset,n_gold_qids,n_gold_labels,missing_required,empty_required,extra_top_level_fields



Source distribution:


,source,n
0,wikidata,1035
1,zero_gold,80
2,kqapro,72
3,mintaka,9



Domain distribution:


,domain,n
0,cinema,181
1,dishes,101
2,people,75
3,kqapro,72
4,countries,71
5,music_albums,69
6,videogames,67
7,paintings,64
8,geo,55
9,museums,53



Complexity distribution:


,complexity,n
0,L3,259
1,L1,239
2,L2,235
3,L4,209
4,L5,93
5,external_multihop,81
6,zero_gold,80


In [10]:
# Inspect a few converted external examples before saving.
preview_cols = ["id", "domain", "complexity", "query_text_ru", "gold_answer_labels_ru"]
preview = pd.DataFrame([
    {col: rec.get(col) for col in preview_cols}
    for rec in merged_records
    if rec.get("constraints", {}).get("source_dataset") in {"zero_gold", "kqapro", "mintaka"}
]).head(20)

display(preview)


,id,domain,complexity,query_text_ru,gold_answer_labels_ru
0,zero_gold_zero_cinema_101,cinema,zero_gold,"Назови 5 фильмов, выпущенных в 1928–1934 годах...",[]
1,zero_gold_zero_cinema_102,cinema,zero_gold,"Подбери 5 сериалов, у которых первый сезон выш...",[]
2,zero_gold_zero_cinema_103,cinema,zero_gold,"Назови 5 документальных фильмов, основанных на...",[]
3,zero_gold_zero_cinema_104,cinema,zero_gold,"Подбери 5 анимационных фильмов, где композитор...",[]
4,zero_gold_zero_cinema_106,cinema,zero_gold,"Подбери 5 художественных фильмов, экранизирующ...",[]
5,zero_gold_zero_cinema_107,cinema,zero_gold,"Назови 5 сериалов, у которых есть хотя бы 8 се...",[]
6,zero_gold_zero_cinema_110,cinema,zero_gold,"Подбери 5 фильмов, получивших «Оскар» за лучши...",[]
7,zero_gold_zero_cinema_111,cinema,zero_gold,"Назови 5 фильмов, где нет ни одного мужчины ср...",[]
8,zero_gold_zero_cinema_112,cinema,zero_gold,"Подбери 5 сериалов, снятых по мотивам комиксов...",[]
9,zero_gold_zero_cinema_113,cinema,zero_gold,"Назови 5 фильмов, которые одновременно являютс...",[]


<a id="save-final-dataset"></a>

## 9. Save final dataset

In [11]:
# Save JSON array as the final merged dataset, plus JSONL for easier line-by-line inspection.
json_path = write_json_array(merged_records, OUTPUT_JSON, overwrite=OVERWRITE_OUTPUT)
jsonl_path = write_jsonl(merged_records, OUTPUT_JSONL, overwrite=OVERWRITE_OUTPUT)

# Diagnostics are also versioned when OVERWRITE_OUTPUT is False.
csv_path = safe_output_path(DIAGNOSTICS_CSV, overwrite=OVERWRITE_OUTPUT)
csv_path.parent.mkdir(parents=True, exist_ok=True)
validation_df.to_csv(csv_path, index=False)

print(f"Saved JSON:  {json_path.relative_to(BASE_DIR)}")
print(f"Saved JSONL: {jsonl_path.relative_to(BASE_DIR)}")
print(f"Saved CSV:   {csv_path.relative_to(BASE_DIR)}")
print(f"Total saved records: {len(merged_records)}")


Saved JSON:  merged_dataset/multihop_benchmark_merged_wikidata_format.json
Saved JSONL: merged_dataset/multihop_benchmark_merged_wikidata_format.jsonl
Saved CSV:   merged_dataset/multihop_benchmark_merged_diagnostics.csv
Total saved records: 1196


## Notes

- `dataset_main.full_gold_enriched.jsonl` defines the target top-level schema. If that file contains extra enriched fields, this notebook preserves them in the final output and fills them with `None` for external sources unless a converter knows how to fill them.
- For `kqapro`, `mintaka`, and `zero_gold`, the full original record is saved inside `constraints["_source_record"]` so no source metadata is lost.
- The notebook does not modify any source files.
